In [ ]:
import climakitae as ck
from climakitae.core.data_interface import get_data
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
import geopandas as gpd
import rioxarray as rxr
import os
import gc
import time
from matplotlib.colors import BoundaryNorm
import matplotlib.cm as cm

# Check for Cartopy installation
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    CARTOPY_AVAILABLE = True
except ImportError:
    print("Cartopy not installed. Visualization requires Cartopy ('pip install cartopy').")
    ccrs = None
    CARTOPY_AVAILABLE = False

# --- Configuration ---
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

plt.rcParams['figure.dpi'] = 150
STANDARD_CRS = "EPSG:4326" # WGS84
DOWNSCALING = "Statistical"
RESOLUTION = "3 km"

# Output directory
output_dir = "OutputImages"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Variables specific to Statistical downscaling
# Using the correct names for the Statistical dataset
VARIABLES_STAT = {
    "T_Max": "Maximum air temperature at 2m",
    "T_Min": "Minimum air temperature at 2m",
    "Precip": "Precipitation (total)"
}

# Scenarios requested for the 2x2 plot
HISTORICAL_SCENARIO = "Historical Climate"
# Note: SSP 1-2.6 is often unavailable for 3km Statistical data and is not included here.
FUTURE_SCENARIOS = ["SSP 2-4.5", "SSP 3-7.0", "SSP 5-8.5"]

# Time periods
# Using the baseline from the original notebook and the requested future period
BASELINE_PERIOD = {"name": "1985-2014", "years": (1985, 2014)}
FUTURE_PERIOD = {"name": "2070-2099", "years": (2070, 2099)}

# --- Load Boundaries ---
SHAPEFILES = {
    "JoshuaTree": "../JoshuaTreeOutlines/JoshuaTree/Joshua_Tree_National_Park.shp",
    "Mojave": "../Mojave/Mojave_National_Preserve.shp"
}

BOUNDARIES = {}
for name, path in SHAPEFILES.items():
    try:
        BOUNDARIES[name] = gpd.read_file(path).to_crs(STANDARD_CRS)
        print(f"Loaded boundary for {name}.")
    except Exception as e:
        print(f"ERROR loading shapefile for {name}: {e}")
        BOUNDARIES[name] = None

# --- Helper Functions ---

def log_progress(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}")

def preprocess_data(dataArray, varKey):
    """Converts units and ensures CRS/Spatial Dims are set correctly."""
    if dataArray is None: return None
    
    # Unit conversions
    if 'T_' in varKey and dataArray.attrs.get('units') == 'K':
        dataArray = dataArray - 273.15
        dataArray.attrs['units'] = '°C'
        
    if varKey == 'Precip' and dataArray.attrs.get('units') in ['kg/m^2/s', 'kg m-2 s-1']:
        days_in_month = dataArray.time.dt.days_in_month
        dataArray = (dataArray * 86400) * days_in_month
        dataArray.attrs['units'] = 'mm/month'

    # Set CRS and Spatial Dims robustly (essential for Statistical data)
    if dataArray.rio.crs is None:
        try: dataArray = dataArray.rio.write_crs(STANDARD_CRS)
        except: pass
            
    y_dim = next((d for d in ['latitude', 'lat', 'y'] if d in dataArray.dims), None)
    x_dim = next((d for d in ['longitude', 'lon', 'x'] if d in dataArray.dims), None)
            
    if y_dim is None or x_dim is None:
        return None
    try:
        return dataArray.rio.set_spatial_dims(x_dim=x_dim, y_dim=y_dim)
    except:
        return None

def calculate_annual_climatology(dataArray, varType):
    """Calculates the annual climatology over the time dimension."""
    # Input dataArray is already time-sliced to the specific period (Baseline or Future)
    if dataArray is None: return None

    if varType == 'Precip':
        # Resample to annual sum (mm/year), then average over the years
        annual_data = dataArray.resample(time='YE').sum(dim='time', skipna=True)
        units = 'mm/year'
    else:
        # Resample to annual mean, then average over the years
        annual_data = dataArray.resample(time='YE').mean(dim='time', skipna=True)
        units = '°C'
        
    annual_climatology = annual_data.mean(dim='time', skipna=True)
    annual_climatology.attrs['units'] = units
    return annual_climatology

def calculate_anomalies(baseline, future, varType):
    """Calculates anomalies (future - baseline)."""
    if baseline is None or future is None: return None

    # Calculate anomaly for each simulation/source_id
    if varType == 'Precip':
        # Percentage change, handle low baseline (1 mm/year threshold)
        epsilon = 1.0 
        anomalies = xr.where(
            abs(baseline) > epsilon,
            ((future - baseline) / baseline) * 100,
            np.nan # Mark as NaN if baseline is too low
        )
        units = '% Change'
    else:
        # Absolute change for temperature
        anomalies = (future - baseline)
        units = f"Δ{baseline.attrs.get('units', '°C')}"
        
    anomalies.attrs['units'] = units
    return anomalies

def reproject_and_clip(dataArray, boundaryGDF):
    """Clips data and ensures standard CRS and dimensions (x/y)."""
    if dataArray is None: return None

    # Ensure CRS matches standard
    if str(dataArray.rio.crs) != STANDARD_CRS:
        try:
            dataArray = dataArray.rio.reproject(STANDARD_CRS)
        except Exception:
            return None

    try:
        # Standardize spatial dimensions to 'x' and 'y'
        x_dim, y_dim = dataArray.rio.x_dim, dataArray.rio.y_dim
        rename_dict = {}
        if x_dim != 'x': rename_dict[x_dim] = 'x'
        if y_dim != 'y': rename_dict[y_dim] = 'y'
        
        if rename_dict:
            dataArray = dataArray.rename(rename_dict)
            dataArray.rio.set_spatial_dims(x_dim='x', y_dim='y', inplace=True)

        # Clipping
        clippedData = dataArray.rio.clip(boundaryGDF.geometry.values, drop=True, all_touched=True)
        clippedData.attrs.update(dataArray.attrs) # Preserve attributes
        return clippedData
    except Exception as e:
        log_progress(f"  Error during clipping: {e}")
        return None

def get_ensemble_mean(dataArray):
    """Calculates the ensemble mean across simulations or source_ids."""
    if dataArray is None: return None
    # Identify the simulation dimension ('source_id' for Statistical, 'simulation' for Dynamical)
    sim_dim = next((d for d in ['simulation', 'source_id'] if d in dataArray.dims), None)
    if sim_dim:
        ensembleMean = dataArray.mean(dim=sim_dim, skipna=True)
        ensembleMean.attrs.update(dataArray.attrs) # Preserve attributes
        return ensembleMean
    return dataArray

# --- Visualization Function ---

def plot_discrete_panel(ax, data, cmap_name, title, projection, plot_extent=None, boundary_gdf=None, num_bins=10, center=None, vmin=None, vmax=None):
    """Helper function for plotting a single panel with a discrete color scale."""
    if data is None:
        ax.set_title(title + "\n(No Data)")
        if projection and plot_extent:
             ax.set_extent(plot_extent, crs=projection)
        return None
        
    # Determine color scale limits if not provided (using 2nd and 98th percentiles)
    if vmin is None or vmax is None:
        try:
            vmin_calc, vmax_calc = np.nanpercentile(data, [2, 98])
            if vmin is None: vmin = vmin_calc
            if vmax is None: vmax = vmax_calc
        except:
            return None

    # Determine bins for discrete scale
    if center is not None:
        # Diverging map (used for anomalies)
        vlim = max(abs(vmin), abs(vmax))
        # Ensure an even number of bins for symmetry
        n_bins = num_bins if num_bins % 2 == 0 else num_bins + 1
        levels = np.linspace(-vlim, vlim, n_bins + 1)
    else:
        # Sequential map (used for baseline)
        levels = np.linspace(vmin, vmax, num_bins + 1)
 
    # Create the colormap and normalization
    try:
        cmap = plt.colormaps[cmap_name].resampled(len(levels) - 1)
    except:
        cmap = plt.get_cmap('viridis', len(levels) - 1) # Fallback

    norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True)
    
    # Plot the data
    mesh = data.plot.pcolormesh(
        ax=ax, transform=projection, cmap=cmap, norm=norm, add_colorbar=False
    )

    if plot_extent and projection:
        ax.set_extent(plot_extent, crs=projection)

    if boundary_gdf is not None:
        # Plot boundary vector over the raster
        boundary_gdf.boundary.plot(ax=ax, color='black', linewidth=1.0, transform=projection)

    ax.set_title(title)
    return mesh


# --- Main Processing Loop (Memory Optimized Strategy) ---

log_progress("=== Starting Spatial Analysis and Visualization ===")

for region_name, boundary_gdf in BOUNDARIES.items():
    if boundary_gdf is None:
        continue

    start_time = time.time()
    log_progress(f"\n--- Processing Region: {region_name} ---")
    
    # Define spatial bounds
    bounds = boundary_gdf.total_bounds
    longitude_slice = (bounds[0], bounds[2])
    latitude_slice = (bounds[1], bounds[3])
    
    # Storage for final processed (clipped, ensemble mean) data for plotting
    final_data = {} 

    # --- Temperature Processing ---
    # Strategy: Fetch periods separately. Process T_Max/T_Min sequentially, calculate climatologies, load, then combine for T_Avg.
    
    log_progress("  [1/2] Analyzing Temperature (T_Avg)...")
    
    temp_climatologies = {} 

    for key in ["T_Max", "T_Min"]:
        varName = VARIABLES_STAT[key]
        
        # 1. Process Baseline Period
        log_progress(f"    Fetching/Analyzing {key} Baseline ({BASELINE_PERIOD['years']})...")
        try:
            # Fetch only the historical scenario for the baseline period
            data_baseline = get_data(
                variable=varName, resolution=RESOLUTION, downscaling_method=DOWNSCALING,
                timescale="monthly", scenario=[HISTORICAL_SCENARIO], time_slice=BASELINE_PERIOD["years"],
                latitude=latitude_slice, longitude=longitude_slice
            )
            data_baseline = preprocess_data(data_baseline, key)
            
            if data_baseline is not None:
                clim_baseline = calculate_annual_climatology(data_baseline, key)
                if clim_baseline is not None: clim_baseline.load() # Load into memory
                temp_climatologies[f"{key}_Baseline"] = clim_baseline
                del data_baseline, clim_baseline
                gc.collect()
        except Exception as e:
            log_progress(f"    Error processing {key} Baseline: {e}")

        # 2. Process Future Period
        log_progress(f"    Fetching/Analyzing {key} Future ({FUTURE_PERIOD['years']})...")
        try:
            # Fetch all future scenarios for the future period
            data_future = get_data(
                variable=varName, resolution=RESOLUTION, downscaling_method=DOWNSCALING,
                timescale="monthly", scenario=FUTURE_SCENARIOS, time_slice=FUTURE_PERIOD["years"],
                latitude=latitude_slice, longitude=longitude_slice
            )
            data_future = preprocess_data(data_future, key)
            
            if data_future is not None:
                clim_future = calculate_annual_climatology(data_future, key)
                if clim_future is not None: clim_future.load() # Load into memory
                temp_climatologies[f"{key}_Future"] = clim_future
                del data_future, clim_future
                gc.collect()
        except Exception as e:
            log_progress(f"    Error processing {key} Future: {e}")

    # Calculate T_Avg Climatologies, Anomalies, Ensemble Mean, and Clip
    log_progress("    Calculating T_Avg Anomalies and Clipping...")
    if temp_climatologies.get("T_Max_Baseline") is not None and temp_climatologies.get("T_Min_Baseline") is not None:
        # T_Avg = (TMax_Clim + TMin_Clim) / 2
        T_Avg_Baseline = (temp_climatologies["T_Max_Baseline"] + temp_climatologies["T_Min_Baseline"]) / 2
        T_Avg_Baseline.attrs['units'] = '°C'

        if temp_climatologies.get("T_Max_Future") is not None and temp_climatologies.get("T_Min_Future") is not None:
             T_Avg_Future = (temp_climatologies["T_Max_Future"] + temp_climatologies["T_Min_Future"]) / 2
             
             # Calculate Anomalies (retains 'scenario' and 'simulation/source_id' dimensions)
             T_Avg_Anomalies = calculate_anomalies(T_Avg_Baseline, T_Avg_Future, "T_Avg")
             
             # Calculate Ensemble Means and Clip
             
             # Baseline (Historical Absolute Value)
             baseline_mean = get_ensemble_mean(T_Avg_Baseline)
             final_data[('T_Avg', HISTORICAL_SCENARIO)] = reproject_and_clip(baseline_mean, boundary_gdf)
             
             # Future Anomalies (by SSP)
             # Calculate ensemble mean across simulations, retaining the 'scenario' dimension
             anomalies_mean = get_ensemble_mean(T_Avg_Anomalies)
             clipped_anomalies = reproject_and_clip(anomalies_mean, boundary_gdf)

             if clipped_anomalies is not None:
                 for ssp in FUTURE_SCENARIOS:
                    try:
                        # Select the specific SSP. Handle potential "Historical + SSP" naming if present.
                        scenario_key = next((s for s in clipped_anomalies.scenario.values if ssp in s), None)
                        if scenario_key:
                             # Squeeze to remove the scenario dimension after selection
                             ssp_data = clipped_anomalies.sel(scenario=scenario_key).squeeze(drop=True)
                             final_data[('T_Avg', ssp)] = ssp_data
                        else:
                            final_data[('T_Avg', ssp)] = None
                    except Exception as e:
                        log_progress(f"    Could not extract {ssp} data: {e}")
                        final_data[('T_Avg', ssp)] = None

    # Clean up temperature intermediates
    del temp_climatologies
    gc.collect()

    # --- Precipitation Processing ---
    log_progress("  [2/2] Analyzing Precipitation...")
    key = "Precip"
    varName = VARIABLES_STAT[key]
    Clim_Baseline = None
    Clim_Future = None

    try:
        # 1. Process Baseline Period
        log_progress(f"    Fetching/Analyzing {key} Baseline...")
        data_baseline = get_data(
            variable=varName, resolution=RESOLUTION, downscaling_method=DOWNSCALING,
            timescale="monthly", scenario=[HISTORICAL_SCENARIO], time_slice=BASELINE_PERIOD["years"],
            latitude=latitude_slice, longitude=longitude_slice
        )
        data_baseline = preprocess_data(data_baseline, key)
        if data_baseline is not None:
            Clim_Baseline = calculate_annual_climatology(data_baseline, key)
            if Clim_Baseline is not None: Clim_Baseline.load()
            del data_baseline
            gc.collect()

        # 2. Process Future Period
        log_progress(f"    Fetching/Analyzing {key} Future...")
        data_future = get_data(
            variable=varName, resolution=RESOLUTION, downscaling_method=DOWNSCALING,
            timescale="monthly", scenario=FUTURE_SCENARIOS, time_slice=FUTURE_PERIOD["years"],
            latitude=latitude_slice, longitude=longitude_slice
        )
        data_future = preprocess_data(data_future, key)
        if data_future is not None:
            Clim_Future = calculate_annual_climatology(data_future, key)
            if Clim_Future is not None: Clim_Future.load()
            del data_future
            gc.collect()
            
        # 3. Calculate Anomalies and Prepare for Plotting
        log_progress("    Calculating Anomalies and Clipping...")
        if Clim_Baseline is not None and Clim_Future is not None:
            Anomalies = calculate_anomalies(Clim_Baseline, Clim_Future, key)
            
            # Baseline (Historical Absolute Value)
            baseline_mean = get_ensemble_mean(Clim_Baseline)
            final_data[('Precip', HISTORICAL_SCENARIO)] = reproject_and_clip(baseline_mean, boundary_gdf)
            
            # Future Anomalies (by SSP)
            anomalies_mean = get_ensemble_mean(Anomalies)
            clipped_anomalies = reproject_and_clip(anomalies_mean, boundary_gdf)
            
            if clipped_anomalies is not None:
                 for ssp in FUTURE_SCENARIOS:
                    try:
                        scenario_key = next((s for s in clipped_anomalies.scenario.values if ssp in s), None)
                        if scenario_key:
                             ssp_data = clipped_anomalies.sel(scenario=scenario_key).squeeze(drop=True)
                             final_data[('Precip', ssp)] = ssp_data
                        else:
                            final_data[('Precip', ssp)] = None
                    except Exception as e:
                        log_progress(f"    Could not extract {ssp} data: {e}")
                        final_data[('Precip', ssp)] = None

            del Clim_Baseline, Clim_Future, Anomalies
            gc.collect()

    except Exception as e:
        log_progress(f"    Error processing {key}: {e}")

    # --- Visualization ---
    if CARTOPY_AVAILABLE and final_data:
        log_progress("  Generating Plots...")
        projection = ccrs.PlateCarree()
        buffer = 0.02
        plot_extent = [longitude_slice[0]-buffer, longitude_slice[1]+buffer,
                       latitude_slice[0]-buffer, latitude_slice[1]+buffer]

        # Define the layout mapping (TL, TR, BL, BR)
        layout = {
            (0, 0): HISTORICAL_SCENARIO,
            (0, 1): "SSP 2-4.5",
            (1, 0): "SSP 3-7.0",
            (1, 1): "SSP 5-8.5"
        }

        # Iterate over variables (Temperature and Precipitation)
        for var_key in ['T_Avg', 'Precip']:
            
            # Check if data exists for this variable
            if (var_key, HISTORICAL_SCENARIO) not in final_data:
                continue

            fig, axes = plt.subplots(2, 2, figsize=(14, 12), subplot_kw={'projection': projection})
            
            # Configure visualization parameters
            if var_key == 'T_Avg':
                cmap_abs = 'plasma'
                cmap_anom = 'RdBu_r' # Diverging: Red=Hotter
                title_main = f'Annual Average Temperature Comparison for {region_name}'
                
            elif var_key == 'Precip':
                cmap_abs = 'GnBu'
                cmap_anom = 'BrBG' # Diverging: Brown=Drier, Green=Wetter
                title_main = f'Annual Precipitation Comparison for {region_name}'

            # Calculate Global Limits for consistent anomaly color scales
            # Combine all anomaly data for this variable
            all_anomaly_data = [final_data.get((var_key, ssp)) for ssp in FUTURE_SCENARIOS if final_data.get((var_key, ssp)) is not None]
            
            if not all_anomaly_data:
                continue
                
            all_anomaly_values = np.concatenate([d.values.flatten() for d in all_anomaly_data])
            
            # Calculate limits (98th percentile of absolute values)
            vlim_anom = np.nanpercentile(np.abs(all_anomaly_values), 98)
            
            # Apply visualization caps if necessary
            if var_key == "Precip" and vlim_anom > 100:
                vlim_anom = 100 # Cap precip visualization at +/- 100%
            
            # Round limits for cleaner colorbar breaks
            if vlim_anom > 1:
                if vlim_anom < 5:
                     vlim_anom = np.ceil(vlim_anom * 2) / 2 # Round to nearest 0.5
                else:
                    round_factor = 5 if vlim_anom < 50 else 10
                    vlim_anom = np.ceil(vlim_anom / round_factor) * round_factor

            meshes = {}
            
            # Plot Baseline (TL - Absolute Scale)
            data_base = final_data.get((var_key, HISTORICAL_SCENARIO))
            # Use center=None for sequential scale
            meshes['base'] = plot_discrete_panel(axes[0, 0], data_base, cmap_abs, 
                                                 f"Historical ({BASELINE_PERIOD['name']})", 
                                                 projection, plot_extent, boundary_gdf, center=None)
            
            # Plot Anomalies (TR, BL, BR - Shared Diverging Scale)
            for idx, ssp in layout.items():
                if ssp == HISTORICAL_SCENARIO: continue
                
                data_anom = final_data.get((var_key, ssp))
                # Use center=0 for diverging scale and the calculated global limits
                mesh = plot_discrete_panel(axes[idx], data_anom, cmap_anom, 
                                            f"{ssp} Anomaly ({FUTURE_PERIOD['name']})", 
                                            projection, plot_extent, boundary_gdf, 
                                            center=0, vmin=-vlim_anom, vmax=vlim_anom)
                if mesh:
                    meshes['anom'] = mesh # Store the last valid mesh for colorbar reference

            # Add Colorbars
            # Adjust layout to make space for colorbars on the right
            fig.subplots_adjust(right=0.85, top=0.9, hspace=0.3, wspace=0.1)

            # Colorbar for Baseline (Top Right)
            if meshes.get('base') and data_base is not None:
                cbar_ax_base = fig.add_axes([0.87, 0.55, 0.02, 0.35]) # [left, bottom, width, height]
                label_base = data_base.attrs.get('units', '')
                fig.colorbar(meshes['base'], cax=cbar_ax_base, orientation='vertical', label=f"Absolute Value ({label_base})")

            # Colorbar for Anomalies (Bottom Right)
            if meshes.get('anom'):
                cbar_ax_anom = fig.add_axes([0.87, 0.1, 0.02, 0.35])
                # Get units from one of the anomaly datasets if available
                data_anom_example = final_data.get((var_key, FUTURE_SCENARIOS[0]))
                label_anom = data_anom_example.attrs.get('units', '') if data_anom_example is not None else ''
                
                # Use 'extend' if the visualization was capped (e.g., Precip > 100%)
                extend_setting = 'both' if (var_key == "Precip" and vlim_anom == 100) else 'neither'
                fig.colorbar(meshes['anom'], cax=cbar_ax_anom, orientation='vertical', label=f"Anomaly ({label_anom})", extend=extend_setting)

            fig.suptitle(title_main, fontsize=16, y=0.98)

            # Save the figure
            save_path = os.path.join(output_dir, f"{region_name}_{var_key}_Spatial_4Panel_Comparison.png")
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            log_progress(f"    Plot saved to {save_path}")
            plt.show()
            
            # Clean up plot resources
            plt.close(fig)

    # Final cleanup for the region
    del final_data
    gc.collect()
    end_time = time.time()
    log_progress(f"--- Finished Region: {region_name}. Time taken: {(end_time - start_time)/60:.2f} minutes ---")

log_progress("=== Analysis Finished ===")

Loaded boundary for JoshuaTree.
Loaded boundary for Mojave.
[22:18:07] === Starting Spatial Analysis and Visualization ===
[22:18:07] 
--- Processing Region: JoshuaTree ---
[22:18:07]   [1/2] Analyzing Temperature (T_Avg)...
[22:18:07]     Fetching/Analyzing T_Max Baseline ((1985, 2014))...
[22:19:19]     Fetching/Analyzing T_Max Future ((2070, 2099))...
WARNING
-------
You have retrieved data for more than one SSP, but not all ensemble members for each GCM are available for all SSPs.

As a result, some scenario and simulation combinations may contain NaN values.

If you want to remove these empty simulations, it is recommended to first subset the data object by each individual scenario and then dropping NaN values.
[22:22:00]     Fetching/Analyzing T_Min Baseline ((1985, 2014))...
[22:23:08]     Fetching/Analyzing T_Min Future ((2070, 2099))...
WARNING
-------
You have retrieved data for more than one SSP, but not all ensemble members for each GCM are available for all SSPs.

As a res